## Bronze transformations

Loads raw csv, sanitizes column names and writes to the `activities_raw` Delta table

In [0]:
catalog = "workouts_demo"
landing_csv = f"/Volumes/{catalog}/bronze/landing/activities.csv"
bronze_tbl = f"{catalog}.bronze.activities_raw"

spark.sql(f"USE CATALOG {catalog}")
spark.sql("USE SCHEMA bronze")

# Robust CSV read for Strava with description line breaks:
df = (spark.read
      .option("header", True)
      .option("multiLine", True)          # <-- allow newlines inside quoted fields
      .option("quote", '"')               # default, being explicit
      .option("escape", '"')              # handle quotes inside quotes
      .option("encoding", "UTF-8")        # adjust if your file uses another encoding
      .option("ignoreLeadingWhiteSpace", True)
      .option("ignoreTrailingWhiteSpace", True)
      .option("mode", "PERMISSIVE")       # keep rows even if some fields are funky
      .option("inferSchema", True)
      # Surface truly broken lines to inspect later
      .option("columnNameOfCorruptRecord", "_corrupt_record")
      .csv(landing_csv))

# Sanitize column names (Delta-safe)
import re
def clean(name: str) -> str:
    name = re.sub(r"[ ,;{}()\n\t=]+", "_", name.strip())
    name = re.sub(r"_+", "_", name).strip("_")
    return name.lower()

df = df.toDF(*[clean(c) for c in df.columns])

# Write to bronze table
(df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(bronze_tbl))
    